# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`  
This notebook provides an end-to-end guide for loading, exploring, and analyzing the FAIR² mass spectrometry dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the FAIR² dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata and create the dataset object
dataset = mlc.Dataset(croissant_url)

# Access and display basic dataset metadata
print(f"\nDataset Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}\n")
print(f"Published: {dataset.metadata.datePublished}")
print(f"Version: {dataset.metadata.version}")
print(f"License: {dataset.metadata.license}")
print(f"Keywords: {getattr(dataset.metadata, 'keywords', None)}\n")

## 2. Data Overview
Explore available record sets, their `@id`s, and their fields. This step helps understand the dataset structure and available fields for further extraction and analysis.

In [ ]:
# List record sets in the dataset by @id
print("Available record sets:'\n-----------------------")
for record_set in dataset.metadata.record_sets:
    print(f"@id: {record_set.id}    name: {getattr(record_set, 'name', '-')}")

# For each record set, print fields and their @id
print("\nFields available in each record set:")
for record_set in dataset.metadata.record_sets:
    print(f"\nRecordSet '@id': {record_set.id} - '{getattr(record_set, 'name', '-')}'")
    for field in record_set.fields:
        print(f"  Field @id: {field.id:40}   name: {getattr(field, 'name', '-')}")

#### Display a few records as preview (using `@id` of a selected record set):

In [ ]:
# Choose the main record set (by @id) for detailed exploration.
# This dataset contains a single main record set named "Mass Spectrometry Results" with a known @id from Croissant.
# We fetch all record set @id values first:
main_record_set_id = dataset.metadata.record_sets[0].id  # If only one

print(f"Previewing first 3 records from record_set @id: {main_record_set_id}\n")
for idx, record in enumerate(dataset.records(record_set=main_record_set_id)):
    print(record)
    if idx >= 2:
        break

## 3. Data Extraction
Load all data from the primary record set into a DataFrame for further analysis.
Use the record set and field `@id`s as shown in the data overview section.

In [ ]:
# Collect all record set @ids
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]

# Dictionary for all dataframes (by record set id)
dataframes = {}
for rs_id in record_set_ids:
    df = pd.DataFrame(dataset.records(record_set=rs_id))
    dataframes[rs_id] = df

print(f"Loaded columns for record_set @id={main_record_set_id}:\n{dataframes[main_record_set_id].columns.tolist()}")
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply basic data processing: filtering, normalization, and grouping.

Below, we select a numeric field (@id) for analysis, filter out low values, normalize, and group by a categorical field.

In [ ]:
# Identify a numeric field by @id -- update these IDs as needed after inspecting available columns
# For demonstration, we'll try typical mass-spec fields
df = dataframes[main_record_set_id]

# Typically, numeric fields: 'cr:field:abundance' or 'cr:field:coverage', here find one:
candidate_numeric_fields = [c for c in df.columns if 'abundance' in c.lower() or 'coverage' in c.lower() or 'peptide' in c.lower() or 'mw' in c.lower()]
print(f"Numeric field candidates: {candidate_numeric_fields}")

# Let's pick the first found for demonstration
numeric_field = candidate_numeric_fields[0] if candidate_numeric_fields else df.columns[0]  # fallback

# Filtering: Remove records with missing or too-low values
threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 10
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with '{numeric_field}' > {threshold:.2f}:")
print(filtered_df[[numeric_field]].head())

# Normalization for the field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized '{numeric_field}' for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Grouping: Try grouping by a typical categorical field, e.g., protein 'description' or 'protein id' if available
group_candidates = [c for c in df.columns if 'description' in c.lower() or 'accession' in c.lower() or 'id' in c.lower()]
group_field = group_candidates[0] if group_candidates else None
if group_field is not None and group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"\nGrouped data by '{group_field}' (mean {numeric_field}):")
    print(grouped_df.head())

## 5. Visualization
Visualize the distribution of the numeric field and relationships, e.g., using a histogram and a boxplot.

Fields are referenced by their `@id` (column names in the DataFrame).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

# Histogram for the numeric field
fig, axs = plt.subplots(1, 2, figsize=(12, 5))
sns.histplot(filtered_df[numeric_field], bins=30, ax=axs[0])
axs[0].set_title(f"Distribution of {numeric_field}")
axs[0].set_xlabel(numeric_field)

# Boxplot for numeric field by group (if group_field present)
if group_field is not None:
    # Take only top 5 groups for visibility
    top_groups = filtered_df[group_field].value_counts().nlargest(5).index
    sns.boxplot(x=group_field, y=numeric_field, data=filtered_df[filtered_df[group_field].isin(top_groups)], ax=axs[1])
    axs[1].set_title(f"{numeric_field} by {group_field} (top 5)")
    axs[1].set_xticklabels(axs[1].get_xticklabels(), rotation=45)
else:
    axs[1].axis('off')
plt.tight_layout()
plt.show()

## 6. Conclusion
In this notebook, we loaded, overviewed, and analyzed the FAIR² human extracellular vesicle mass spectrometry dataset using `mlcroissant`.  
Key findings include:
- Listing and understanding available record sets and fields by their `@id`s for robust referencing and reproducibility.
- Filtering and normalization of numeric fields (e.g., abundance or coverage), and categorical grouping.
- Basic visualizations of value distributions and group comparisons.

This approach facilitates downstream FAIR data science, such as reproducible biomarker analysis and cross-study comparison.